# 03 — Build the Gold Layer

Now we build the analytics layer:
1. **Dynamic Tables** (SILVER) — cleaned, typed, auto-refreshing
2. **Views** (GOLD) — star schema dimensions and facts
3. **Semantic View** — business-friendly model for Cortex Analyst
4. **Cortex Agent** — natural language Q&A over the fleet data

In [ ]:
-- Set context
SET user_db = 'HYDRAB_HOL_' || CURRENT_USER();
USE DATABASE IDENTIFIER($user_db);
USE WAREHOUSE HYDRAB_HOL_WH;

## SILVER: Dynamic Tables

Dynamic Tables auto-refresh from BRONZE. We set a 5-minute target lag.

In [ ]:
-- Vehicles Silver: clean vehicle master
CREATE OR REPLACE DYNAMIC TABLE SILVER.VEHICLES_SILVER
  TARGET_LAG = '5 minutes'
  WAREHOUSE = HYDRAB_HOL_WH
AS
SELECT
  "Chassis_Number__c" AS vin,
  "Product_Name__c" AS model,
  "Account_Name__c" AS customer,
  "Depot_Name__c" AS depot,
  "Status" AS status,
  "Install_Date" AS install_date
FROM BRONZE.ASSET
WHERE "Chassis_Number__c" IS NOT NULL;

In [ ]:
-- Telemetry Silver: parsed Odos events
CREATE OR REPLACE DYNAMIC TABLE SILVER.TELEMETRY_SILVER
  TARGET_LAG = '5 minutes'
  WAREHOUSE = HYDRAB_HOL_WH
AS
SELECT
  RAW:vin::STRING AS vin,
  RAW:timestamp::TIMESTAMP AS event_time,
  RAW:signals:battery_soc::FLOAT AS battery_soc,
  RAW:signals:speed::FLOAT AS speed_kmh,
  RAW:signals:battery_temperature::FLOAT AS cell_temp,
  RAW:signals:latitude::FLOAT AS latitude,
  RAW:signals:longitude::FLOAT AS longitude,
  RAW:signals:odometer::FLOAT AS odometer_km,
  RAW:signals:energy_consumed::FLOAT AS energy_kwh
FROM BRONZE.ODOS_EVENTS;

In [ ]:
-- Defects Silver
CREATE OR REPLACE DYNAMIC TABLE SILVER.DEFECTS_SILVER
  TARGET_LAG = '5 minutes'
  WAREHOUSE = HYDRAB_HOL_WH
AS
SELECT
  "Asset__c" AS asset_id,
  "Defect_Type__c" AS defect_type,
  "Severity__c" AS severity,
  "Root_Cause__c" AS root_cause,
  "Repair_Cost__c" AS repair_cost,
  "CreatedDate" AS created_date
FROM BRONZE.DEFECT_EVENT;

## GOLD: Star Schema Views

In [ ]:
USE SCHEMA GOLD;

-- Dimension: Vehicle
CREATE OR REPLACE VIEW DIM_VEHICLE AS
SELECT * FROM SILVER.VEHICLES_SILVER;

-- Fact: Latest Telemetry per Vehicle
CREATE OR REPLACE VIEW FCT_LATEST_TELEMETRY AS
SELECT *
FROM SILVER.TELEMETRY_SILVER
QUALIFY ROW_NUMBER() OVER (PARTITION BY vin ORDER BY event_time DESC) = 1;

-- Fact: Defects
CREATE OR REPLACE VIEW FCT_DEFECT AS
SELECT * FROM SILVER.DEFECTS_SILVER;

## Semantic View

The Semantic View defines a business-friendly model for Cortex Analyst.

In [ ]:
CREATE OR REPLACE CORTEX SEMANTIC VIEW GOLD.FLEET_SEMANTIC
AS SEMANTIC MODEL $$
name: fleet_analytics
description: HydraB electric bus fleet analytics - vehicles, telemetry, defects

tables:
  - name: vehicles
    base_table:
      database: '{{DATABASE}}'
      schema: GOLD
      table: DIM_VEHICLE
    dimensions:
      - name: vin
        expr: vin
        description: Vehicle Identification Number
      - name: model
        expr: model
        description: Bus model name
      - name: customer
        expr: customer
        description: Transit operator
      - name: depot
        expr: depot
        description: Home depot location
      - name: status
        expr: status
        description: Vehicle status
    metrics:
      - name: total_vehicles
        expr: COUNT(DISTINCT vin)
        description: Total number of vehicles
      - name: vehicles_by_customer
        expr: COUNT(DISTINCT vin)
        description: Vehicle count grouped by customer

  - name: telemetry
    base_table:
      database: '{{DATABASE}}'
      schema: GOLD
      table: FCT_LATEST_TELEMETRY
    dimensions:
      - name: vin
        expr: vin
      - name: event_time
        expr: event_time
    metrics:
      - name: avg_soc
        expr: AVG(battery_soc)
        description: Average battery state of charge
      - name: avg_speed
        expr: AVG(speed_kmh)
        description: Average speed in km/h
      - name: max_temperature
        expr: MAX(cell_temp)
        description: Maximum cell temperature
      - name: low_battery_count
        expr: COUNT_IF(battery_soc < 20)
        description: Vehicles with SOC below 20%

  - name: defects
    base_table:
      database: '{{DATABASE}}'
      schema: GOLD
      table: FCT_DEFECT
    dimensions:
      - name: defect_type
        expr: defect_type
      - name: severity
        expr: severity
      - name: root_cause
        expr: root_cause
    metrics:
      - name: total_defects
        expr: COUNT(*)
        description: Total defect count
      - name: avg_repair_cost
        expr: AVG(repair_cost)
        description: Average repair cost in GBP
      - name: critical_defects
        expr: COUNT_IF(severity = 'Critical')
        description: Number of critical severity defects
$$;

## Cortex Agent

The Agent combines structured analytics (via Semantic View) with natural language Q&A.

In [ ]:
CREATE OR REPLACE CORTEX AGENT GOLD.FLEET_AGENT
  COMMENT = 'HydraB Fleet Intelligence Agent'
AS
  USING (
    CORTEX_ANALYST(
      SEMANTIC_VIEW => 'GOLD.FLEET_SEMANTIC'
    )
  );

In [ ]:
-- Test the agent
SELECT SNOWFLAKE.CORTEX.INVOKE_AGENT(
  'GOLD.FLEET_AGENT',
  'Which bus models have the most defects and what is their average repair cost?'
) AS agent_response;

## Gold Layer Complete!

You now have:
- Auto-refreshing Dynamic Tables in SILVER
- Star schema views in GOLD
- A Semantic View for natural language analytics
- A Cortex Agent you can ask questions

**Next:** Open `04_deploy_dashboard.ipynb` to deploy the React dashboard.